# Development environment setup check

Verifies that this machine is ready to run the pipeline: Conda, the three named
environments (`clamp-analyses`, `gpu-kmeans`, `snakemake`), Snakemake itself, and the
data files each rule expects to already exist on disk. Read-only diagnostic - it does
not install, download, or build anything. Run `setup.sh` first to create the envs;
see `data/README.md` / `data/gtex/README.md` / `data/pseudobulk/README` for the
manual data downloads this notebook checks for.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from pyprojroot import here


## Settings

In [ ]:
ROOT = Path(here())

with open(ROOT / 'workflow/config/gtex.yaml') as handle:
    GTEX_CONFIG = yaml.safe_load(handle)['gtex']
with open(ROOT / 'workflow/config/pseudobulk.yaml') as handle:
    PSEUDOBULK_CONFIG = yaml.safe_load(handle)

REQUIRED_ENVS = {
    'clamp-analyses': ROOT / 'envs/clamp-analyses.yaml',
    'gpu-kmeans': ROOT / 'envs/gpu-kmeans.yaml',
    'snakemake': ROOT / 'envs/snakemake.yaml',
}


## Helpers

In [ ]:
def run(cmd, cwd=None, timeout=60):
    """subprocess.run wrapper that never raises -- a missing tool/env is a check
    result, not a notebook-crashing error."""
    try:
        return subprocess.run(
            cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout
        )
    except Exception as exc:  # FileNotFoundError, TimeoutExpired, ...
        return subprocess.CompletedProcess(cmd, returncode=1, stdout='', stderr=str(exc))


def conda_env_names():
    result = run(['conda', 'env', 'list', '--json'])
    if result.returncode != 0:
        return set()
    try:
        envs = json.loads(result.stdout).get('envs', [])
    except json.JSONDecodeError:
        return set()
    return {Path(p).name for p in envs}


def conda_run(env, *args, timeout=60):
    """Run a command inside a named conda env, returning stripped stdout or None."""
    result = run(['conda', 'run', '-n', env, *args], timeout=timeout)
    return result.stdout.strip() if result.returncode == 0 else None


def human_size(num_bytes):
    size = float(num_bytes)
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if size < 1024:
            return f'{size:.0f}{unit}' if unit == 'B' else f'{size:.1f}{unit}'
        size /= 1024
    return f'{size:.1f}PB'


def path_status(path):
    """('present'/'missing', note) for a single path, without recursing into large dirs."""
    p = Path(path)
    if not p.exists():
        return 'missing', ''
    if p.is_dir():
        return 'present', 'directory'
    return 'present', human_size(p.stat().st_size)


## Check: Conda

In [ ]:
rows = []

conda_path = shutil.which('conda')
if conda_path:
    version = run(['conda', '--version']).stdout.strip()
    rows.append({'category': 'conda', 'item': 'conda', 'status': 'OK', 'note': f'{version} ({conda_path})'})
else:
    rows.append({'category': 'conda', 'item': 'conda', 'status': 'MISSING', 'note': 'run ./setup.sh to install Miniconda'})

present_envs = conda_env_names() if conda_path else set()


## Check: conda environments

In [ ]:
PIN_CHECKS = {
    'clamp-analyses': [('python', ['python', '--version']), ('R', ['Rscript', '-e', 'cat(R.version.string)'])],
    'gpu-kmeans': [('python', ['python', '--version'])],
    'snakemake': [('python', ['python', '--version']), ('snakemake', ['snakemake', '--version'])],
}

for env_name, env_file in REQUIRED_ENVS.items():
    if env_name not in present_envs:
        rows.append({
            'category': 'conda-env', 'item': env_name, 'status': 'MISSING',
            'note': f'run ./setup.sh (spec: {env_file.relative_to(ROOT)})',
        })
        continue
    for label, cmd in PIN_CHECKS.get(env_name, []):
        value = conda_run(env_name, *cmd)
        status = 'OK' if value else 'WARN'
        note = value if value else f'could not run `{" ".join(cmd)}` in env'
        rows.append({'category': 'conda-env', 'item': f'{env_name}: {label}', 'status': status, 'note': note})

# CLAMP R package: external dependency pinned by scripts/install_clamp.R.
# A package with no RemoteSha, or a different SHA, is not considered compatible.
if 'clamp-analyses' in present_envs:
    clamp_check = run(
        ['conda', 'run', '-n', 'clamp-analyses', 'Rscript',
         'scripts/install_clamp.R', '--check'], cwd=ROOT,
    )
    clamp_version = conda_run(
        'clamp-analyses', 'Rscript', '-e',
        "cat(as.character(packageVersion('CLAMP')))",
    )
    if clamp_check.returncode == 0:
        pin_note = clamp_check.stdout.strip()
        rows.append({'category': 'conda-env', 'item': 'clamp-analyses: pinned CLAMP revision', 'status': 'OK', 'note': f'v{clamp_version}; {pin_note}'})
    else:
        rows.append({
            'category': 'conda-env', 'item': 'clamp-analyses: pinned CLAMP revision', 'status': 'MISMATCH',
            'note': clamp_check.stderr.strip() or 'run ./setup.sh to install the required revision',
        })


## Check: Snakemake

In [ ]:
if 'snakemake' in present_envs:
    dry_run = run(
        ['conda', 'run', '-n', 'snakemake', 'snakemake', '-n',
         '--snakefile', 'workflow/Snakefile', 'all'],
        cwd=ROOT, timeout=180,
    )
    if dry_run.returncode == 0:
        rows.append({'category': 'snakemake', 'item': 'DAG dry-run (rule `all`)', 'status': 'OK', 'note': 'workflow/Snakefile parses, rule graph builds'})
    else:
        tail = (dry_run.stdout + dry_run.stderr).strip().splitlines()[-5:]
        rows.append({'category': 'snakemake', 'item': 'DAG dry-run (rule `all`)', 'status': 'FAIL', 'note': ' | '.join(tail)})
else:
    rows.append({'category': 'snakemake', 'item': 'DAG dry-run (rule `all`)', 'status': 'SKIPPED', 'note': "snakemake env missing, see above"})


## Check: required data files

Built from `workflow/config/gtex.yaml` / `workflow/config/pseudobulk.yaml` rather than
hardcoded, so it can't drift from what the Snakemake rules actually consume. Mirrors
`dataset_raw_inputs()` / `pseudobulk_path()` in `workflow/rules/pseudobulk.smk`.

In [ ]:
required = []


def add_required(category, item, path):
    required.append({'category': category, 'item': item, 'path': str(ROOT / path)})


# GTEx
add_required('auto (snakemake download_gtex_raw)', 'GTEx raw bulk TPM GCT', GTEX_CONFIG['raw_gz'])
add_required('manual', 'GTEx sample-attributes metadata', GTEX_CONFIG['metadata'])
add_required('manual (repo-tracked)', 'GTEx anatomical subtissue spec', GTEX_CONFIG['subtissue_inference']['anatomical_spec'])

# Shared references
refs = PSEUDOBULK_CONFIG['references']
add_required('auto (snakemake pathway_prior)', 'GO-BP pathway prior GMT', refs['go_bp_file'])
add_required('manual', 'Cell marker reference (Cell_marker_Human.xlsx)', refs['cell_marker_file'])
add_required('manual', 'Allen Brain Atlas GMT', refs['allen_brain_gmt_file'])

# Pseudobulk datasets: raw single-cell inputs + pre-built pseudobulk products
for dataset, cfg in PSEUDOBULK_CONFIG['datasets'].items():
    raw_keys = ['raw']
    if cfg['kind'] == 'matrix_market':
        raw_keys += ['features', 'barcodes', 'metadata']
    for key in raw_keys:
        add_required('manual', f'{dataset} raw: {key}', cfg[key])

    if not cfg.get('build_pseudobulk', False):
        pb_dir = cfg['pseudobulk_dir']
        files = ['bulk_expr.csv', 'patient_info.csv', 'truthFrac_v0.csv']
        if 'truth_v1_col' in cfg:
            files.append('truthFrac_v1.csv')
        for filename in files:
            add_required('manual', f'{dataset} pre-built: {filename}', f'{pb_dir}/{filename}')

for entry in required:
    status, note = path_status(entry['path'])
    if status == 'missing':
        row_status = 'MISSING (auto)' if entry['category'].startswith('auto') else 'MISSING'
        note = f"see README.md / data/README.md" if row_status == 'MISSING' else f"run `snakemake --use-conda {entry['category'].split()[-1].rstrip(')')}`"
    else:
        row_status = 'OK'
    rows.append({'category': entry['category'], 'item': entry['item'], 'status': row_status, 'note': note})


## Check: optional / legacy data (not wired into any Snakemake rule)

These are read directly by standalone notebooks (ARCHS4, recount2, PhenomeXcan,
drug-disease associations, a few GTEx-biology-notebook-only files) rather than being
Snakemake rule inputs, per `data/README.md`. Missing entries here are informational,
not failures.

In [ ]:
LEGACY_DATA = [
    ('data/archs4/human_gene_v2.5.h5', 'ARCHS4 human gene expression matrix'),
    ('data/recount2/recount2_PLIER_data', 'recount2 PLIER-prep data (dir)'),
    ('data/phenomexcan', 'PhenomeXcan S-PrediXcan/S-MultiXcan results (dir)'),
    ('data/drug_disease_associations', 'Drug-disease associations (dir)'),
    ('data/gtex/GTEx_Analysis_v8_xCell_scores_7_celltypes.txt', 'GTEx xCell scores'),
    ('data/gtex/GSE198623_human_processed.h5ad', 'GSE198623 processed h5ad'),
    ('data/gtex/GTEx_8_tissues_snRNAseq_atlas_071421.public_obs.h5ad', 'GTEx 8-tissue snRNA-seq atlas'),
    ('data/gtex/phenomexcan_simplified_phenotypes_info.tsv.gz', 'PhenomeXcan simplified phenotypes'),
    ('data/gtex/phenoplier', 'PhenoPLIER GLS summary (dir)'),
    ('data/pathways', 'Extra pathway/geneset files beyond GO-BP (dir)'),
]

for rel_path, label in LEGACY_DATA:
    status, note = path_status(ROOT / rel_path)
    row_status = 'OK' if status == 'present' else 'not present (optional)'
    rows.append({'category': 'optional-legacy', 'item': label, 'status': row_status, 'note': note})


## Summary

In [ ]:
summary = pd.DataFrame(rows, columns=['category', 'item', 'status', 'note'])

ok_mask = summary['status'].isin(['OK'])
warn_mask = summary['status'].str.startswith('not present') | (summary['status'] == 'WARN')
fail_mask = ~(ok_mask | warn_mask)

print(f"{ok_mask.sum()} OK, {warn_mask.sum()} optional/warn, {fail_mask.sum()} missing/failed")
if fail_mask.any():
    print('\nNeeds attention:')
    print(summary.loc[fail_mask, ['item', 'status', 'note']].to_string(index=False))

summary
